# Atlas cluster evidence

Close the phase 3 integration comparison and screen Harmony clusters for shared or disease-associated candidates.

Inputs:
- postprocessed atlas or 100k sample from [`run_atlas_postprocessing.py`](../../pipelines/run_atlas_postprocessing.py)
- disease labels from [`disease_markers`](../../scripts/disease_markers/README.md)

Development and screening use the local 100k sample. Final abundance and pseudobulk DE should be re-run on the full atlas on the server.

In [ ]:
from pathlib import Path

import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from cytetype import rank_genes_groups_backed
from disease_markers.candidates import (
    annotate_obs_with_labels,
    classify_cluster_candidates,
    cluster_support_table,
    same_study_contrast_support,
)
from disease_markers.concordance import (
    cluster_label_purity,
    concordance_summary,
    source_label_contingency,
)
from disease_markers.labels import build_sample_label_table
from disease_markers.validation import (
    de_supported_candidates,
    differential_abundance_by_study,
    filter_pseudobulk_profiles,
    sample_cluster_proportions,
    shared_direction_genes,
)
from metadata.config import MetadataConfig
from shared.repo import REPO_ROOT

sc.set_figure_params(figsize=(3.5, 3.5), frameon=False)

REPO = REPO_ROOT
HARMONY_H5AD = REPO / "output/atlas/v2/post/production/atlas_v2_post_sample.h5ad"
CONTEXTS = REPO / "output/context/contexts_v2.jsonl"
ATLAS_CSV = REPO / "output/atlas/v2/atlas_v2.csv"
SAMPLE_METADATA = MetadataConfig().sampleParquetPath
OUT_DIR = REPO / "output/atlas/v2/analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY = "SRX_accession"
STUDY_KEY = "study_accession"
CLUSTER_KEY = "leiden_atlas"
LABEL_KEY = "cell_type"
ONTOLOGY_KEY = "cell_ontology_term_id"
DISEASE_NAME_KEY = "diseaseName"
DISEASE_ONTOLOGY_KEY = "disease_ontology_term_id"
MIN_CELLS_PER_PROFILE = 10


In [ ]:
# sc.pl.embedding(atlas, "X_umap_uncorrected", color="cell_type")
# sc.pl.embedding(atlas, "X_umap", color="cell_type")
# (
#     adjusted_mutual_info_score(atlas.obs["cell_type"], atlas.obs["leiden_uncorrected"]),
#     adjusted_mutual_info_score(atlas.obs["cell_type"], atlas.obs["leiden_atlas"])
# )



In [ ]:
# Load into memory. backed="r" makes raw.X a _CSRDataset, and raw.to_adata() fails on .copy().
# Labels are joined in this same cell so later analysis cannot run on an unlabeled atlas.
harmony = sc.read_h5ad(HARMONY_H5AD)
if harmony.raw is None:
    raise ValueError("Expected full-gene counts in adata.raw")

print(f"Loaded {harmony.n_obs:,} cells x {harmony.n_vars:,} HVGs; raw {harmony.raw.n_vars:,} genes")
atlas = harmony.raw.to_adata()
for col in harmony.obs.columns:
    atlas.obs[col] = harmony.obs[col].values
for key in harmony.obsm.keys():
    atlas.obsm[key] = harmony.obsm[key]
assert atlas.n_vars == harmony.raw.n_vars
print(f"Full-gene atlas ready: {atlas.n_obs:,} cells x {atlas.n_vars:,} genes")

label_table = build_sample_label_table(CONTEXTS, ATLAS_CSV, SAMPLE_METADATA)
label_table.to_csv(OUT_DIR / "sample_labels.csv", index=False)

atlas.obs = annotate_obs_with_labels(atlas.obs, label_table, sampleKey=SAMPLE_KEY, eligibleOnly=False)
eligible = atlas.obs["eligible"].fillna(False).astype(bool)
atlas = atlas[eligible.to_numpy()].copy()
missing_labels = [
    c
    for c in ("diseaseArea", "diseased", DISEASE_NAME_KEY, DISEASE_ONTOLOGY_KEY)
    if c not in atlas.obs.columns
]
if missing_labels:
    raise RuntimeError(f"Label join failed; missing columns: {missing_labels}")
print(
    f"Eligible cells: {atlas.n_obs:,}; samples: {atlas.obs[SAMPLE_KEY].nunique()}; "
    f"studies: {atlas.obs[STUDY_KEY].nunique()}; clusters: {atlas.obs[CLUSTER_KEY].nunique()}"
)
print(label_table["diseaseAreaSource"].value_counts(dropna=False).to_string())
atlas.obs[["diseaseArea", DISEASE_NAME_KEY, "diseased", "controlType", DISEASE_ONTOLOGY_KEY]].head()


In [ ]:
# Disease-label diagnostics: prefer human-readable MONDO names for display.
sample_level = label_table.copy()
print("diseaseAreaSource counts:")
print(sample_level["diseaseAreaSource"].value_counts(dropna=False).to_string())
print("\ndiseaseArea counts:")
print(sample_level["diseaseArea"].value_counts(dropna=False).to_string())
print(
    "\nMONDO coverage:",
    sample_level["diseaseOntologyTermId"].fillna("").astype(str).str.strip().ne("").sum(),
    "/",
    len(sample_level),
)
print("\nTop diseaseName values:")
print(sample_level["diseaseName"].fillna("(missing)").value_counts().head(15).to_string())

from sklearn.metrics import adjusted_mutual_info_score

sc.pl.umap(atlas, color=[DISEASE_NAME_KEY, "diseaseArea"], ncols=1)
(
    adjusted_mutual_info_score(
        atlas.obs[CLUSTER_KEY].astype(str),
        atlas.obs[DISEASE_NAME_KEY].fillna("missing").astype(str),
    ),
    adjusted_mutual_info_score(
        atlas.obs[CLUSTER_KEY].astype(str),
        atlas.obs["diseaseArea"].astype(str),
    ),
)


In [ ]:
# from sklearn.metrics import adjusted_mutual_info_score

# # sc.pl.umap(atlas, color=["diseaseArea", "controlType", "cell_type"], ncols=1)
# atlas.obs["diseased"] = atlas.obs["diseased"].astype(str)
# atlas.obs["isBiologicalControl"] = atlas.obs["isBiologicalControl"].astype(str)
# atlas.obs["eligible"] = atlas.obs["eligible"].astype(str)
# atlas.obs["controlType"] = atlas.obs["controlType"].astype(str)

# cols = ["leiden_atlas", "cell_type", "diseaseArea", "diseased", "controlType", "isBiologicalControl", "eligible"]

# sc.pl.umap(atlas, color=cols, ncols=1)
# # (
# #     adjusted_mutual_info_score(atlas.obs["leiden_atlas"], atlas.obs["cell_type"]),
# #     adjusted_mutual_info_score(atlas.obs["leiden_atlas"], atlas.obs["diseaseArea"]),
# # )

# for col in cols[1:]:
#     print(col, adjusted_mutual_info_score(atlas.obs["leiden_atlas"], atlas.obs[col]))

In [ ]:
# Labels are now joined in the load cell above.
# Keep this cell as a guard so a stale kernel state fails loudly.
required = {"diseaseArea", "diseased", "eligible"}
missing = sorted(required - set(atlas.obs.columns))
if missing:
    raise RuntimeError(
        "atlas.obs is missing disease labels. Re-run the load cell so labels are joined before analysis."
    )
print("Disease labels present on atlas.obs")
atlas.obs[["diseaseArea", "diseased", "controlType"]]#.head()


## Phase 3 closeout: integrated clusters versus source labels

Preserved `cell_type` and `cell_ontology_term_id` values come from individually processed source datasets. They are weak references, not ground truth. Concordance plus existing scIB and corrected/uncorrected UMAPs is the operational comparison to individual processing.

In [ ]:
contingency = source_label_contingency(atlas.obs, clusterKey=CLUSTER_KEY, labelKey=LABEL_KEY)
concordance_cell_type = concordance_summary(
    atlas.obs, clusterKey=CLUSTER_KEY, labelKey=LABEL_KEY, studyKey=STUDY_KEY
)
concordance_ontology = concordance_summary(
    atlas.obs, clusterKey=CLUSTER_KEY, labelKey=ONTOLOGY_KEY, studyKey=STUDY_KEY
)
purity = cluster_label_purity(
    atlas.obs, clusterKey=CLUSTER_KEY, labelKey=LABEL_KEY, studyKey=STUDY_KEY
)

contingency.to_csv(OUT_DIR / "source_label_contingency.csv")
concordance_cell_type.to_csv(OUT_DIR / "source_label_concordance.csv", index=False)
concordance_ontology.to_csv(OUT_DIR / "source_ontology_concordance.csv", index=False)
purity.to_csv(OUT_DIR / "cluster_label_purity.csv", index=False)

concordance_cell_type.loc[concordance_cell_type["scope"].isin(["global", "studyMacro"])]


## Cluster candidate screen

Three separate concepts:
- **shared candidate**: many studies and disease areas, not dominated by one study
- **disease-associated candidate**: enough same-study case/control profiles for at least one area
- **unexpected convergence**: broad study support with mixed source labels

Do not call a cluster a biological state until marker programs are coherent.

In [ ]:
if "diseaseArea" not in atlas.obs.columns or "diseased" not in atlas.obs.columns:
    raise RuntimeError(
        "atlas.obs is missing disease labels. Re-run the load cell so labels are joined before analysis."
    )

support = cluster_support_table(
    atlas.obs,
    clusterKey=CLUSTER_KEY,
    sampleKey=SAMPLE_KEY,
    studyKey=STUDY_KEY,
    labelKey=LABEL_KEY,
    ontologyKey=ONTOLOGY_KEY,
)
contrast_support = same_study_contrast_support(
    atlas.obs,
    clusterKey=CLUSTER_KEY,
    sampleKey=SAMPLE_KEY,
    studyKey=STUDY_KEY,
    minCellsPerProfile=MIN_CELLS_PER_PROFILE,
)
candidates = classify_cluster_candidates(support, contrast_support)

support.to_csv(OUT_DIR / "cluster_support.csv", index=False)
contrast_support.to_csv(OUT_DIR / "same_study_contrast_support.csv", index=False)
candidates.to_csv(OUT_DIR / "cluster_candidates.csv", index=False)

candidates.loc[
    candidates["isAnyCandidate"],
    [
        "cluster",
        "nCells",
        "nStudies",
        "nDiseaseAreas",
        "dominantStudyFraction",
        "topSourceLabel",
        "topSourceLabelFraction",
        "isSharedCandidate",
        "isDiseaseAssociatedCandidate",
        "isUnexpectedConvergenceCandidate",
    ],
].sort_values(["isDiseaseAssociatedCandidate", "nStudies"], ascending=[False, False])


## Integrated cluster marker screen

Rank markers for each Harmony cluster against all remaining eligible cells. This uses the log-normalized 2,000-HVG matrix in `harmony.X`; `harmony.raw.X` contains counts and is not suitable for this ranking. These markers support cluster interpretation and marker-program review, but they do not replace the same-study disease-versus-control pseudobulk analysis below.

In [ ]:
MARKER_KEY = f"rank_genes_{CLUSTER_KEY}"
N_MARKERS_PER_CLUSTER = 100
MIN_CELLS_PER_MARKER_CLUSTER = 50

# Exclude small clusters before ranking because their one-vs-rest markers are
# unstable and rank_genes_groups_backed rejects clusters containing one cell.
marker_cluster_sizes = atlas.obs[CLUSTER_KEY].astype(str).value_counts()
ranked_marker_clusters = marker_cluster_sizes[
    marker_cluster_sizes >= MIN_CELLS_PER_MARKER_CLUSTER
].index
skipped_marker_clusters = (
    marker_cluster_sizes[marker_cluster_sizes < MIN_CELLS_PER_MARKER_CLUSTER]
    .rename_axis("cluster")
    .reset_index(name="nCells")
)
marker_cell_mask = atlas.obs[CLUSTER_KEY].astype(str).isin(ranked_marker_clusters)
marker_obs_names = atlas.obs_names[marker_cell_mask]
if len(ranked_marker_clusters) < 2:
    raise RuntimeError("Marker ranking requires at least two clusters above the cell-count gate")
print(
    f"Ranking {len(ranked_marker_clusters)} clusters; skipped "
    f"{len(skipped_marker_clusters)} with fewer than {MIN_CELLS_PER_MARKER_CLUSTER} cells"
)

# Build a lightweight object from the log-normalized HVG matrix. Avoid copying
# harmony.raw because it contains the much larger full-gene count matrix.
marker_positions = harmony.obs_names.get_indexer(marker_obs_names)
if np.any(marker_positions < 0):
    raise RuntimeError("Eligible marker-screen cells are missing from the Harmony object")
marker_obs = harmony.obs.iloc[marker_positions][[CLUSTER_KEY]].copy()
marker_obs[CLUSTER_KEY] = marker_obs[CLUSTER_KEY].astype("category").cat.remove_unused_categories()
marker_atlas = AnnData(
    X=harmony.X[marker_positions].copy(),
    obs=marker_obs,
    var=harmony.var.copy(),
)
marker_atlas.uns["log1p"] = dict(harmony.uns.get("log1p", {}))

rank_genes_groups_backed(
    marker_atlas,
    groupby=CLUSTER_KEY,
    use_raw=False,
    n_genes=N_MARKERS_PER_CLUSTER,
    key_added=MARKER_KEY,
    pts=True,
)
cluster_markers = sc.get.rank_genes_groups_df(marker_atlas, group=None, key=MARKER_KEY).rename(
    columns={
        "group": "cluster",
        "names": "gene",
        "scores": "score",
        "logfoldchanges": "log2FoldChange",
        "pvals": "pValue",
        "pvals_adj": "adjustedPValue",
        "pct_nz_group": "fractionInCluster",
        "pct_nz_reference": "fractionOutsideCluster",
    }
)
cluster_markers["cluster"] = cluster_markers["cluster"].astype(str)
cluster_markers["rankWithinCluster"] = cluster_markers.groupby("cluster", observed=True).cumcount() + 1
cluster_markers["detectionDifference"] = (
    cluster_markers["fractionInCluster"] - cluster_markers["fractionOutsideCluster"]
)
cluster_markers = cluster_markers.merge(
    candidates[
        [
            "cluster",
            "topSourceLabel",
            "isSharedCandidate",
            "isDiseaseAssociatedCandidate",
            "isUnexpectedConvergenceCandidate",
            "isAnyCandidate",
        ]
    ],
    on="cluster",
    how="left",
)
cluster_markers.to_csv(OUT_DIR / "cluster_markers.csv", index=False)

candidate_marker_review = cluster_markers[
    cluster_markers["isAnyCandidate"].fillna(False)
    & (cluster_markers["rankWithinCluster"] <= 10)
].copy()
candidate_marker_review

## Study-aware abundance validation

Abundance uses per-SRX cluster proportions and same-study case/control differences. Cells are never treated as independent replicates.

In [ ]:
proportions = sample_cluster_proportions(
    atlas.obs,
    clusterKey=CLUSTER_KEY,
    sampleKey=SAMPLE_KEY,
    studyKey=STUDY_KEY,
)
proportions.to_csv(OUT_DIR / "sample_cluster_proportions.csv", index=False)

disease_candidates = candidates.loc[candidates["isDiseaseAssociatedCandidate"], "cluster"].astype(str)
eligible_contrasts = contrast_support[
    contrast_support["cluster"].astype(str).isin(set(disease_candidates))
    & contrast_support["eligibleForContrast"].astype(bool)
].copy()

abundance_frames: list[pd.DataFrame] = []
for row in eligible_contrasts.itertuples(index=False):
    frame = differential_abundance_by_study(
        proportions,
        area=str(row.diseaseArea),
        cluster=str(row.cluster),
        minSamplesPerArm=2,
    )
    if not frame.empty:
        abundance_frames.append(frame)

abundance = pd.concat(abundance_frames, ignore_index=True) if abundance_frames else pd.DataFrame()
abundance.to_csv(OUT_DIR / "differential_abundance_by_study.csv", index=False)
abundance.head()


## Study-aware pseudobulk DE and shared programs

Pseudobulks are SRX x cluster sums. Contrasts are restricted to studies that contain both cases and controls for the tested disease area. The design includes study when multiple overlapping studies are available. Distinct disease areas are never pooled into one case class.

On the 100k sample this is a smoke test. Re-run against the full atlas before claiming disease programs.

In [ ]:
pb_result = dc.pp.pseudobulk(
    atlas,
    sample_col=SAMPLE_KEY,
    groups_col=CLUSTER_KEY,
    mode="sum",
)
pdata = pb_result[0] if isinstance(pb_result, tuple) else pb_result
pdata = filter_pseudobulk_profiles(pdata, minCellsPerProfile=MIN_CELLS_PER_PROFILE)

labels_by_srx = label_table.set_index("srxAccession")
pdata.obs["diseaseArea"] = pdata.obs[SAMPLE_KEY].map(labels_by_srx["diseaseArea"]).astype("category")
pdata.obs["diseased"] = pdata.obs[SAMPLE_KEY].map(labels_by_srx["diseased"]).astype("boolean")
pdata.obs[STUDY_KEY] = pdata.obs[SAMPLE_KEY].map(labels_by_srx["studyAccession"])
print(f"Pseudobulk profiles after min-cell filter: {pdata.n_obs:,}")
pdata.obs.head()


In [ ]:
de_summary, de_hits, de_results = de_supported_candidates(
    pdata,
    candidates,
    contrast_support,
    clusterKey=CLUSTER_KEY,
    studyKey=STUDY_KEY,
)
shared_genes = shared_direction_genes(de_hits, minDiseaseAreas=2)

de_summary.to_csv(OUT_DIR / "de_summary.csv", index=False)
de_hits.to_csv(OUT_DIR / "de_hits.csv", index=False)
de_results.to_csv(OUT_DIR / "de_results.csv", index=False)
shared_genes.to_csv(OUT_DIR / "shared_direction_genes.csv", index=False)

de_summary


In [ ]:
def _volcano_on_axes(
    ax: plt.Axes,
    results: pd.DataFrame,
    title: str,
    *,
    padj_threshold: float = 0.05,
    lfc_threshold: float = 1.0,
) -> None:
    if results.empty:
        ax.set_title(f"{title}\n(skipped)", fontsize=8)
        ax.axis("off")
        return
    frame = results.copy()
    frame["negLog10Padj"] = -np.log10(frame["padj"].clip(lower=1e-300))
    sig = (
        frame["padj"].notna()
        & (frame["padj"] <= padj_threshold)
        & (frame["log2FoldChange"].abs() >= lfc_threshold)
    )
    ax.scatter(frame.loc[~sig, "log2FoldChange"], frame.loc[~sig, "negLog10Padj"], s=4, alpha=0.3, c="0.6")
    ax.scatter(frame.loc[sig, "log2FoldChange"], frame.loc[sig, "negLog10Padj"], s=8, alpha=0.8, c="C0")
    ax.axhline(-np.log10(padj_threshold), color="0.3", lw=0.8, ls="--")
    ax.axvline(lfc_threshold, color="0.3", lw=0.8, ls="--")
    ax.axvline(-lfc_threshold, color="0.3", lw=0.8, ls="--")
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("log2FC")
    ax.set_ylabel("-log10 padj")

def plot_volcanoes_by_cluster(de_results: pd.DataFrame, de_summary: pd.DataFrame) -> None:
    if de_summary.empty or "cluster" not in de_results.columns:
        print("No DE results to plot")
        return
    fig_dir = OUT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    for cluster in sorted(de_summary["cluster"].astype(str).unique(), key=str):
        area_rows = de_summary[de_summary["cluster"].astype(str) == cluster]
        areas = area_rows["diseaseArea"].astype(str).tolist()
        n = max(len(areas), 1)
        fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.0), squeeze=False)
        for ax, area in zip(axes[0], areas, strict=True):
            mask = (de_results["cluster"].astype(str) == cluster) & (
                de_results["diseaseArea"].astype(str) == area
            )
            _volcano_on_axes(ax, de_results.loc[mask], area)
        fig.suptitle(f"Cluster {cluster}", fontsize=11, y=1.02)
        fig.tight_layout()
        fig.savefig(fig_dir / f"volcano_cluster_{cluster}.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

plot_volcanoes_by_cluster(de_results, de_summary)
shared_genes.head(20)


In [ ]:
# Optional pathway activity on DE hits with Decoupler resource nets.
# Keep this exploratory; shared-direction genes above are the primary cross-disease summary.
if not de_hits.empty:
    try:
        net = dc.op.progeny(organism="human", top=100)
        mat = (
            de_hits.pivot_table(
                index=["cluster", "diseaseArea"],
                columns="gene",
                values="stat",
                aggfunc="first",
            )
            .fillna(0.0)
        )
        pathway = dc.mt.ulm(mat=mat, net=net)
        pathway_df = pathway if isinstance(pathway, pd.DataFrame) else pathway[0]
        pathway_df.to_csv(OUT_DIR / "pathway_ulm.csv")
        pathway_df.head()
    except Exception as exc:
        print(f"Pathway scoring skipped: {exc}")
else:
    print("No DE hits for pathway scoring")


## Artifact checklist

Written under `output/atlas/v2/analysis/`:
- `source_label_concordance.csv`, `source_ontology_concordance.csv`, `cluster_label_purity.csv`
- `cluster_support.csv`, `cluster_candidates.csv`, `same_study_contrast_support.csv`
- `cluster_markers.csv`
- `sample_cluster_proportions.csv`, `differential_abundance_by_study.csv`
- `de_summary.csv`, `de_hits.csv`, `de_results.csv`, `shared_direction_genes.csv`

Limitations of the local 100k sample: no sampling provenance in `uns`, rare clusters under-represented, and same-study contrasts are thinner than on the full atlas.